# Word2Vec: Efficient Estimation of Word Representations in Vector Space (Mikolov et al., ICLR 2013)
### Triển Khai Bằng PyTorch & Thử Nghiệm Đối Chiếu Giá Trị Học Thuật

> **Tác giả:** Tomas Mikolov, Kai Chen, Greg Corrado, Jeffrey Dean  
> **Hội nghị:** ICLR 2013 (Scottsdale, Arizona)  
> **Ý nghĩa lịch sử:** Khai sinh khái niệm Word Embeddings hiện đại, đưa ngôn ngữ tự nhiên từ các mã One-Hot rời rạc vào không gian hình học liên tục có thể thực hiện đại số vector ($King - Man + Woman \approx Queen$).

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Skip-Gram với Negative Sampling:** Triển khai mô hình bằng hai tầng `nn.Embedding` và hàm mất mát `F.logsigmoid` tối ưu hóa cao.
2. **Đại số ngữ nghĩa:** Thực thi phép suy luận vector $v(\text{King}) - v(\text{Man}) + v(\text{Woman}) \approx v(\text{Queen})$.
3. **Trực quan hóa không gian nhúng 2D:** Sử dụng phân tích thành phần chính (PCA) để minh họa các cụm từ vựng gom nhóm tự nhiên theo trường nghĩa.
4. **Academic Value Benchmark:** Đối chuẩn 3 mô hình học thuật:
   - *Mô hình 1 (Kỷ nguyên Tiền-Word2Vec):* Ma trận đồng xuất hiện kết hợp phân rã SVD $\rightarrow$ Cồng kềnh, phân rã tĩnh.
   - *Mô hình 2 (Word2Vec Full Softmax):* Điểm nghẽn tính toán $O(V \times D)$ $\rightarrow$ Không thể mở rộng quy mô lớn.
   - *Mô hình 3 (Word2Vec Negative Sampling):* Đột phá $O(K \times D)$ $\rightarrow$ Nhanh gấp hàng chục lần, bảo toàn đại số ngữ nghĩa.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Thiết bị tính toán được chọn: {device}")

## PHẦN 1: QUẢN LÝ DỮ LIỆU TẬP VĂN BẢN PYTORCH

Tạo các cặp `(center_word, context_word)` kèm $K=5$ mẫu âm tính ngẫu nhiên rút từ bảng phân phối lũy thừa $3/4$.

In [ ]:
from word2vec_pytorch import TextCorpusDataset

corpus = """
king rules the royal empire with the queen
queen is the royal woman ruler beside the king
man is a royal king who rules the court
woman is a royal queen who rules the court
apple is a fresh sweet fruit from the garden
orange is a juicy citrus fruit from the garden
banana is a yellow sweet fruit from the garden
paris is the beautiful capital city of france
rome is the historic capital city of italy
hanoi is the charming capital city of vietnam
tokyo is the modern capital city of japan
"""

dataset = TextCorpusDataset(corpus, window_size=3, num_negatives=5)
print(f"[*] Kích thước từ điển (Vocab Size): {dataset.vocab_size}")
print(f"[*] Tổng số cặp ngữ cảnh sinh ra:  {len(dataset)}")

## PHẦN 2: KIẾN TRÚC MÔ HÌNH SKIP-GRAM & FULL SOFTMAX

- **SkipGramModel:** Dùng hai ma trận nhúng `in_embeddings` và `out_embeddings`. Mất mát Negative Sampling:
$$\mathcal{L} = -\log \sigma(v_{\text{pos}} \cdot v_{\text{center}}) - \sum_{k=1}^K \log \sigma(-v_{\text{neg}, k} \cdot v_{\text{center}})$$
- **FullSoftmaxModel:** Tính Cross-Entropy trên toàn bộ không gian $V$ nhãn từ điển.

In [ ]:
from word2vec_pytorch import SkipGramModel, FullSoftmaxModel

model_neg = SkipGramModel(dataset.vocab_size, embed_dim=32).to(device)
model_softmax = FullSoftmaxModel(dataset.vocab_size, embed_dim=32).to(device)
print("[✓] Đã khởi tạo thành công SkipGramModel và FullSoftmaxModel.")

## PHẦN 3: BỘ GIẢI ĐẠI SỐ TƯƠNG ĐỒNG (ANALOGY SOLVER)

Tìm từ $w^*$ sao cho vector của nó gần nhất với $\vec{v}(\text{king}) - \vec{v}(\text{man}) + \vec{v}(\text{woman})$ theo độ tương đồng Cosine.

In [ ]:
from word2vec_pytorch import solve_analogy_pytorch
print("[✓] Đã sẵn sàng hàm solve_analogy_pytorch.")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

So sánh trực tiếp:
1. **Mô Hình 1: Co-occurrence Matrix + SVD (Kỷ nguyên Tiền-Word2Vec)**
2. **Mô Hình 2: Word2Vec Full Softmax (Điểm nghẽn $O(V)$)**
3. **Mô Hình 3: Word2Vec Negative Sampling (Đột phá $O(K)$)**

In [ ]:
from word2vec_pytorch import run_academic_value_benchmark

# Chạy kiểm thử đối chuẩn học thuật
bench_results = run_academic_value_benchmark(device)

## PHẦN 5: TRỰC QUAN HÓA KHÔNG GIAN NHÚNG 2D (PCA PROJECTION)

Chiếu không gian vector 32 chiều xuống 2D bằng PCA. Các từ cùng trường ngữ nghĩa (Hoàng gia, Trái cây, Thủ đô - Quốc gia) sẽ tự động hình thành các cụm riêng biệt.

In [ ]:
from word2vec_pytorch import plot_embedding_space

out_dir = "outputs"
plot_embedding_space(bench_results["model_neg"], bench_results["dataset"], output_dir=out_dir)

# Hiển thị ảnh không gian vector
embed_img_path = os.path.join(out_dir, "word2vec_embedding_space.png")
if os.path.exists(embed_img_path):
    img_embed = plt.imread(embed_img_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img_embed)
    plt.axis('off')
    plt.title("Không Gian Nhúng Vector 2D Của Word2Vec", fontsize=14, fontweight='bold')
    plt.show()

## PHẦN 6: BIỂU ĐỒ ĐỐI CHIẾU HỌC THUẬT CHUYÊN SÂU

Xuất biểu đồ 4 bảng phân tích:
1. **Đường cong suy giảm mất mát:** Negative Sampling vs Full Softmax.
2. **Quy luật mở rộng độ phức tạp:** $O(V)$ bùng nổ theo kích thước từ điển vs $O(K)$ giữ hằng số.
3. **Đại số vector ngữ nghĩa:** Mô phỏng các hệ số vector trong phép toán $King - Man + Woman = Queen$.
4. **Ma trận tương đồng Cosine:** Kiểm chứng các khái niệm liên quan có độ tương đồng cao vượt trội.

In [ ]:
from word2vec_pytorch import plot_comprehensive_academic_comparison

plot_comprehensive_academic_comparison(bench_results, output_dir=out_dir)

# Hiển thị biểu đồ đối chiếu
comp_img_path = os.path.join(out_dir, "word2vec_academic_comparison.png")
if os.path.exists(comp_img_path):
    img_comp = plt.imread(comp_img_path)
    plt.figure(figsize=(16, 12))
    plt.imshow(img_comp)
    plt.axis('off')
    plt.title("Kết Quả Đối Chiếu Học Thuật Toàn Diện (Word2Vec ICLR 2013)", fontsize=14, fontweight='bold')
    plt.show()

## 🎓 KẾT LUẬN & Ý NGHĨA LỊCH SỬ CỦA WORD2VEC (ICLR 2013)

1. **Bước nhảy vọt từ Rời rạc sang Liên tục:** Xóa sổ hoàn toàn cách biểu diễn One-Hot Encoding cồng kềnh, đặt nền móng cho toàn bộ các kiến trúc biểu diễn ngôn ngữ hiện đại.
2. **Negative Sampling:** Một phát kiến thuật toán thiên tài, giải phóng mô hình khỏi điểm nghẽn tính toán $O(V)$, cho phép Google huấn luyện trên hàng trăm tỷ từ vựng báo chí (Google News) trong thời gian ngắn.
3. **Đại số hình học ngôn ngữ:** Khám phá chấn động rằng các quan hệ ngữ nghĩa tinh tế của con người có thể được mô phỏng bằng các phép cộng trừ vector đơn giản trong không gian Euclidean.
4. **Di sản bất tử:** Từ GloVe, FastText đến các tầng Token Embeddings trong Transformer, BERT và GPT ngày nay đều kế thừa trực tiếp nguyên lý nền tảng từ Word2Vec.